# Apoio para reinicio da primeira carga em desenvolvimento

Este notebook possui duas operacoes independentes:

1. `mostrar_estado_desenvolvimento`: somente leitura das tabelas Hive, dos temporarios HDFS, das origens conhecidas no CTL e das tabelas Oracle configuradas.
2. `limpar_estado_desenvolvimento`: limpeza somente do sandbox Hive `sbx_t2i2016` e dos dois temporarios HDFS.

Nenhuma tabela Oracle e apagada por este notebook.

In [ ]:
# ============================================================
# CONEXAO SPARK DE APOIO
# ============================================================

import os
from traceback import format_exc

from src.utils.gerenciador_sessao_spark_local import GerenciadorSessaoSpark

AMBIENTE_ALVO = "MODELAGEM"
DOMINIO_ALVO = "t2i"
SANDBOX_ALVO = "t2i2016"
DATABASE_ALVO = f"sbx_{SANDBOX_ALVO}"
PATH_SAVE_HDFS_ALVO = f"/dados/transientes/{DOMINIO_ALVO}/{SANDBOX_ALVO}"

ambiente_local = os.environ.get("AMBIENTE", "").strip().upper()
if ambiente_local != AMBIENTE_ALVO:
    raise RuntimeError(
        f"Este notebook exige AMBIENTE=MODELAGEM; encontrado={ambiente_local or '<ausente>'}."
    )

spark = None
gerenciador_spark = GerenciadorSessaoSpark(
    nome_sessao="apoio-reset-desenvolvimento",
    adicionar_variaveis={
        "DOMINIO": DOMINIO_ALVO,
        "AMBIENTE": AMBIENTE_ALVO,
        "SANDBOX": SANDBOX_ALVO,
    },
    exibir_configuracao=False,
    ativar_logs=True,
)

spark = gerenciador_spark.criar_sessao_spark(
    db2=True,
    driver_memory="12g",
    driver_cores=4,
    executor_memory="12g",
    executor_cores=4,
    num_executors=8,
    jars=["/dados/shared/bin/ojdbc8.jar"],
    spark_conf={
        "spark.driver.memoryOverhead": "8g",
        "spark.executor.memoryOverhead": "4g",
        "spark.serializer": "org.apache.spark.serializer.KryoSerializer",
        "spark.kryoserializer.buffer.max": "512m",
        "spark.sql.adaptive.enabled": "true",
        "spark.sql.adaptive.coalescePartitions.enabled": "true",
        "spark.sql.adaptive.skewJoin.enabled": "true",
        "spark.sql.shuffle.partitions": "240",
        "spark.sql.sources.partitionOverwriteMode": "dynamic",
        "spark.hadoop.mapreduce.input.fileinputformat.input.dir.recursive": "true",
        "spark.sql.autoBroadcastJoinThreshold": "-1",
        "spark.sql.broadcastTimeout": "8000",
        "spark.executor.heartbeatInterval": "30s",
        "spark.network.timeout": "300s",
        "spark.sql.session.timeZone": "America/Sao_Paulo",
    },
)

print(f"Spark conectado. ambiente={AMBIENTE_ALVO} database={DATABASE_ALVO} path_hdfs={PATH_SAVE_HDFS_ALVO}")

In [ ]:
# ============================================================
# CONEXAO REMOTA E FUNCOES MINIMAS LOCAIS
# ============================================================

if spark is None:
    raise RuntimeError("A sessao Spark de apoio nao foi criada.")

%run ./src/utils/gerenciador_sessao_spark_remoto.ipynb

In [ ]:
%%spark

# ============================================================
# CONFIGURACAO E INSPECAO SOMENTE LEITURA
# ============================================================

import os

AMBIENTE_APOIO = "MODELAGEM"
DOMINIO_APOIO = "t2i"
SANDBOX_APOIO = "t2i2016"
DATABASE_APOIO = f"sbx_{SANDBOX_APOIO}"
PATH_SAVE_HDFS_APOIO = f"/dados/transientes/{DOMINIO_APOIO}/{SANDBOX_APOIO}"
CONFIRMACAO_LIMPEZA = "LIMPAR_SBX_T2I2016"

TABELAS_HIVE_APOIO = [
    "CTL_OPRL_RCM",
    "RCM_FNC_CLI",
    "RCM_FNC_CLI_EVTL",
    "ANDO_RCM_FNC_CLI",
    "ANDO_RCM_FNC_CLI_EVTL",
]

TABELAS_ORACLE_APOIO = [
    "RCM_VLDD",
    "RCM_VRS",
    "AVS_SELD",
    "PBCO_CADD",
    "SGT_CADD",
    "AVS_FNC_CLI",
]

def validar_identificador_sql(valor, nome: str) -> str:
    final = str(valor or "").strip()
    if not final or not final.replace("_", "a").isalnum() or final[0].isdigit():
        raise ValueError(f"Identificador SQL invalido para {nome}: {valor}")
    return final

def nome_tabela_hive(database: str, tabela: str) -> str:
    return f"{validar_identificador_sql(database, 'database')}.{validar_identificador_sql(tabela, 'tabela')}"

def nome_tabela_hive_sql(database: str, tabela: str) -> str:
    return f"`{validar_identificador_sql(database, 'database')}`.`{validar_identificador_sql(tabela, 'tabela')}`"

def tabela_hive_existe(database: str, tabela: str) -> bool:
    database_final = validar_identificador_sql(database, "database")
    tabela_final = validar_identificador_sql(tabela, "tabela")
    return bool(spark.catalog.tableExists(tabela_final, database_final))

def ler_tabela_hive(database: str, tabela: str):
    return spark.sql(f"SELECT * FROM {nome_tabela_hive_sql(database, tabela)}")

def caminhos_temporarios_ctl(path_save_hdfs: str) -> dict:
    raiz = str(path_save_hdfs or "").rstrip("/")
    if not raiz:
        raise ValueError("path_save_hdfs nao pode ser vazio.")
    return {
        "temp_backup": f"{raiz}/temp_backup_CTL_OPRL_RCM",
        "temp_lineage": f"{raiz}/temp_lineage_CTL_OPRL_RCM",
    }

def path_hdfs_existe(path_hdfs: str) -> bool:
    conf = spark._jsc.hadoopConfiguration()
    path = spark._jvm.org.apache.hadoop.fs.Path(path_hdfs)
    fs = path.getFileSystem(conf)
    return bool(fs.exists(path))

def remover_path_hdfs_se_existir(path_hdfs: str, executar: bool, logger_etapa=None, contexto: str = "HDFS") -> bool:
    existe = path_hdfs_existe(path_hdfs)
    if not existe:
        if logger_etapa is not None:
            logger_etapa.info(f"[{contexto}][HDFS] Path inexistente; limpeza ja concluida. path={path_hdfs}")
        return True
    if not executar:
        if logger_etapa is not None:
            logger_etapa.info(f"[{contexto}][DRY_RUN] Path seria removido se executar=True. path={path_hdfs}")
        return False
    conf = spark._jsc.hadoopConfiguration()
    path = spark._jvm.org.apache.hadoop.fs.Path(path_hdfs)
    fs = path.getFileSystem(conf)
    removido = bool(fs.delete(path, True))
    if logger_etapa is not None:
        if removido:
            logger_etapa.info(f"[{contexto}][HDFS] Path removido. path={path_hdfs}")
        else:
            logger_etapa.error(f"[{contexto}][HDFS] Falha ao remover path. path={path_hdfs}")
    return removido

def _validar_alvo_desenvolvimento():
    if AMBIENTE_APOIO != "MODELAGEM":
        raise RuntimeError("Operacao permitida somente em MODELAGEM.")
    if DATABASE_APOIO != "sbx_t2i2016":
        raise RuntimeError(f"Database de apoio invalido: {DATABASE_APOIO}")
    if not DATABASE_APOIO.startswith("sbx_"):
        raise RuntimeError(f"Database fora de sandbox: {DATABASE_APOIO}")

def _mostrar_hive(tabela: str, amostra: int = 5):
    objeto = nome_tabela_hive(DATABASE_APOIO, tabela)
    if not tabela_hive_existe(DATABASE_APOIO, tabela):
        print(f"[HIVE][AUSENTE] {objeto}")
        return

    df = ler_tabela_hive(DATABASE_APOIO, tabela)
    quantidade = df.count()
    print(f"\n[HIVE][TABELA] {objeto}\n[HIVE][QTD] {quantidade}")
    df.printSchema()
    print(f"[HIVE][AMOSTRA] {objeto}")
    df.limit(amostra).show(truncate=False)

def _mostrar_origens_do_ctl():
    tabela_ctl = "CTL_OPRL_RCM"
    if not tabela_hive_existe(DATABASE_APOIO, tabela_ctl):
        return

    df_ctl = ler_tabela_hive(DATABASE_APOIO, tabela_ctl)
    colunas = {col.upper() for col in df_ctl.columns}
    if {"NM_DB_OGM", "NM_TAB_OGM"} - colunas:
        return

    origens = (
        df_ctl.select("NM_DB_OGM", "NM_TAB_OGM")
        .where("NM_DB_OGM IS NOT NULL AND NM_TAB_OGM IS NOT NULL")
        .distinct()
        .collect()
    )

    if not origens:
        print("\n[ORIGEM][INFO] CTL sem origens cadastradas.")
        return

    print("\n[ORIGEM][CTL] Origens distintas encontradas na CTL")
    for linha in origens:
        database_origem = validar_identificador_sql(str(linha["NM_DB_OGM"]), "database_origem")
        tabela_origem = validar_identificador_sql(str(linha["NM_TAB_OGM"]), "tabela_origem")
        objeto = f"{database_origem}.{tabela_origem}"
        try:
            df_origem = spark.sql(f"SELECT * FROM `{database_origem}`.`{tabela_origem}`")
            print(f"[ORIGEM][OK] {objeto} qtd={df_origem.count()}")
            df_origem.limit(5).show(truncate=False)
        except Exception as exc:
            print(f"[ORIGEM][ERRO] {objeto} motivo={str(exc)}")

cliente_oracle_apoio = None
oracle_schema_apoio = None

def _obter_cliente_oracle_apoio():
    global cliente_oracle_apoio, oracle_schema_apoio
    if cliente_oracle_apoio is None:
        cliente_oracle_apoio = criar_cliente_oracle_spark(env=dict(os.environ))
        oracle_schema_apoio = cliente_oracle_apoio.schema
    return cliente_oracle_apoio, oracle_schema_apoio

def _mostrar_oracle():
    try:
        cliente, owner = _obter_cliente_oracle_apoio()
        owner = validar_identificador_sql(owner, "oracle_schema")
        print(f"\n[ORACLE][SCHEMA] {owner} (somente leitura)")
    except Exception as exc:
        print(f"[ORACLE][CONEXAO_ERRO] {str(exc)}")
        return

    for tabela in TABELAS_ORACLE_APOIO:
        objeto = f"{owner}.{tabela}"
        try:
            df_qtd = cliente.run_select(f"SELECT COUNT(1) AS QTD FROM {objeto}")
            quantidade = df_qtd.first()[0]
            df_amostra = cliente.run_select(f"SELECT * FROM {objeto} WHERE ROWNUM <= 5")
            print(f"\n[ORACLE][TABELA] {objeto}\n[ORACLE][QTD] {quantidade}")
            df_amostra.printSchema()
            df_amostra.show(truncate=False)
        except Exception as exc:
            print(f"[ORACLE][ERRO] {objeto} motivo={str(exc)}")

def mostrar_estado_desenvolvimento(incluir_oracle: bool = True):
    _validar_alvo_desenvolvimento()
    print(f"[APOIO][CONTEXTO] ambiente={AMBIENTE_APOIO} database={DATABASE_APOIO} path={PATH_SAVE_HDFS_APOIO}")
    print("[APOIO][HIVE] Tabelas de desenvolvimento")
    for tabela in TABELAS_HIVE_APOIO:
        _mostrar_hive(tabela)
    _mostrar_origens_do_ctl()

    caminhos = caminhos_temporarios_ctl(PATH_SAVE_HDFS_APOIO)
    print("\n[APOIO][HDFS] Temporarios")
    for nome, caminho in caminhos.items():
        print(f"[HDFS][{nome}] existe={path_hdfs_existe(caminho)} path={caminho}")

    if incluir_oracle:
        _mostrar_oracle()

print("Funcoes de apoio carregadas. A limpeza ainda nao foi executada.")

In [ ]:
%%spark

# ============================================================
# OPERACAO 1: INSPECAO SOMENTE LEITURA
# ============================================================

mostrar_estado_desenvolvimento(incluir_oracle=True)

In [ ]:
%%spark

# ============================================================
# OPERACAO 2: LIMPEZA HIVE/HDFS
# ============================================================
# Esta funcao NAO limpa Oracle. A chamada esta em uma celula
# separada e exige a confirmacao textual exata.

def limpar_estado_desenvolvimento(confirmacao: str):
    _validar_alvo_desenvolvimento()
    if confirmacao != CONFIRMACAO_LIMPEZA:
        raise ValueError(
            f"Confirmacao invalida. Digite exatamente: {CONFIRMACAO_LIMPEZA}"
        )

    print(f"[LIMPEZA][INICIO] alvo={DATABASE_APOIO}")
    print("[LIMPEZA][ORACLE] nenhuma operacao de escrita sera executada no Oracle.")

    for tabela in TABELAS_HIVE_APOIO:
        objeto = nome_tabela_hive_sql(DATABASE_APOIO, tabela)
        if not tabela_hive_existe(DATABASE_APOIO, tabela):
            print(f"[LIMPEZA][HIVE][AUSENTE] {DATABASE_APOIO}.{tabela}")
            continue
        print(f"[LIMPEZA][HIVE][TRUNCATE] {DATABASE_APOIO}.{tabela}")
        spark.sql(f"TRUNCATE TABLE {objeto}")

    caminhos = caminhos_temporarios_ctl(PATH_SAVE_HDFS_APOIO)
    for nome, caminho in caminhos.items():
        removido = remover_path_hdfs_se_existir(
            caminho,
            executar=True,
            logger_etapa=logger,
            contexto="APOIO_LIMPEZA",
        )
        if not removido:
            raise RuntimeError(f"Nao foi possivel limpar o temporario HDFS {nome}: {caminho}")

    print("[LIMPEZA][FIM] Hive e temporarios HDFS limpos; Oracle preservado.")

In [ ]:
%%spark

# ============================================================
# CHAMADA DE LIMPEZA - EXECUCAO MANUAL
# ============================================================
# Descomente somente depois de conferir a inspeção acima.
# limpar_estado_desenvolvimento("LIMPAR_SBX_T2I2016")